# CarePath DARAG — 00 Data prep  `[CPU runtime]`
Setup + datastore + real ASR pairs + supplementary labeled pairs (former stages
00–03). All CPU — run on a free Colab CPU runtime. Each step resumes, so re-running
after a disconnect skips finished work.

In [ ]:
# --- CarePath stage bootstrap (short by design) ---
import importlib.util, os, shutil, subprocess, sys
from pathlib import Path

def _find(start):
    for d in [start, *start.parents]:
        if (d / 'pyproject.toml').exists() and (d / 'apps' / 'api' / 'carepath').exists():
            return d
    return None

def _token():
    # Colab Secrets live in userdata, NOT os.environ - check both.
    for key in ('CAREPATH_GITHUB_TOKEN', 'GITHUB_TOKEN'):
        if os.environ.get(key):
            return os.environ[key]
    try:
        from google.colab import userdata
        for key in ('CAREPATH_GITHUB_TOKEN', 'GITHUB_TOKEN'):
            try:
                val = userdata.get(key)
                if val:
                    return val
            except Exception:
                pass
    except Exception:
        pass
    return None

REPO = _find(Path.cwd().resolve())
if REPO is None and importlib.util.find_spec('google.colab'):
    target = Path('/content/carepath')
    if _find(target):                       # already cloned in this runtime
        REPO = target
    else:
        if target.exists():
            shutil.rmtree(target)           # remove a half-cloned leftover
        url = os.environ.get('CAREPATH_REPO_URL', 'https://github.com/truong-tt/carepath.git')
        tok = _token()
        if tok and url.startswith('https://github.com/'):
            url = url.replace('https://', f'https://x-access-token:{tok}@')
        r = subprocess.run(['git', 'clone', url, str(target)], capture_output=True, text=True)
        if r.returncode != 0:
            err = (r.stderr or r.stdout)
            if tok:
                err = err.replace(tok, '***')
            raise SystemExit(
                'git clone failed. This repo is private — add a Colab Secret named '
                'GITHUB_TOKEN (key icon in the left sidebar, toggle "Notebook access") '
                'holding a GitHub token with read access to the repo, then re-run.\n\n' + err)
        REPO = target
assert REPO, 'Open this notebook from inside the CarePath repo.'
os.chdir(REPO); sys.path.insert(0, str(REPO / 'apps' / 'api'))

PROFILE = os.environ.get('CAREPATH_PROFILE', 'full')  # default full; set CAREPATH_PROFILE=smoke for a plumbing-only check
from carepath.gec.notebook import init_stage
CTX = init_stage(PROFILE); P = CTX.paths; PROF = CTX.profile


In [ ]:
# Install the GEC training stack (idempotent; needed once per Colab runtime).
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[training]'])


# CarePath DARAG — Stage 00: Setup & Config  `[CPU]`
Bootstrap the repo, pick a **profile** (`smoke` plumbing check / `full` ViMedCSS
run), mount Drive on Colab, and print the resolved paths. Every later stage reuses
this `init_stage` so run-sizes and artifact paths have one source of truth.

In [ ]:
from carepath.gec import env
print(env.gpu_report())
print('dataset    ->', CTX.dataset)
print('datastore  ->', P.datastore)
print('real pairs ->', P.real_pairs)
print('adapters   ->', P.adapters)
print('serve dir  ->', P.serve_bundle)

# --- Colab Pro runtime plan (which runtime per notebook) ---
print('''
Run the four notebooks in order, each on the runtime it names:
  00_data_prep        [CPU]        datastore + real/labeled pairs (free)
  01_synthesis        [GPU L4]     synthetic transcripts + TTS + pairs + leakage
  02_train_predict    [GPU A100]   harvest + augment + QLoRA train + predict
  03_evaluate_export  [CPU]        WER/NE-F1 tables + gate + serve bundle
Full run ~ 3 seeds x 4 variants; training dominates GPU units, viXTTS is the other
long pole. Every step --resumes, so a disconnect continues, it does not restart.
Set CAREPATH_PROFILE=full for the real ViMedCSS run (smoke = plumbing only).''')


# Stage 01: Build the NE / code-switch datastore  `[CPU]`
Paper §4.2 Step 1 — union the curated lexicon, dataset `cs_terms_list`, and mined
code-switch tokens into the retrieval datastore.

In [ ]:
CTX.run_step(['scripts/gec/build_datastore.py', '--dataset', CTX.dataset,
              '--limit-per-split', str(PROF.limit_per_split or 0), '--output', str(P.datastore)])
import json
print('terms:', json.load(open(P.datastore, encoding='utf-8'))['metadata']['term_count'])


# Stage 02: Real GEC pairs + N-best + error-signal report  `[CPU]`
Paper §3.1 — run Gipformer (or mock for smoke) over ViMedCSS audio to build
`raw_asr -> gold_text` pairs. `--n-best` adds the perturbation hypotheses (paper
§4.3). The **error-signal report** warns if the ASR is too accurate on train to
teach the corrector (paper §3.2).

In [ ]:
CTX.run_step(['scripts/gec/make_pairs.py', '--dataset', CTX.dataset, '--output', str(P.real_pairs),
              '--asr-provider', PROF.asr_provider, '--datastore', str(P.datastore),
              '--retrieval-backend', PROF.retrieval_backend,
              '--limit-per-split', str(PROF.limit_per_split or 0),
              '--n-best', str(PROF.n_best), '--resume'])
from carepath.gec.data import read_jsonl
from carepath.gec.evaluate import train_error_signal
print(train_error_signal(read_jsonl(P.real_pairs)))
CTX.save([str(P.datastore), str(P.real_pairs)])


# Stage 03: Supplementary real pairs from the Label Studio export  `[CPU]`
The clinician-corrected export is already a real `raw_asr -> gold_text` pair
(Whisper draft + human edit). Ingest it as supplementary real data (ViMedCSS stays
primary). Skipped automatically until the labeling workflow has produced rows.

In [ ]:
from pathlib import Path
export = 'data/labeling/training_transcripts.jsonl'
if Path(export).exists():
    CTX.run_step(['scripts/gec/make_labeled_pairs.py', '--input', export,
                  '--output', str(P.labeled_pairs), '--datastore', str(P.datastore),
                  '--retrieval-backend', PROF.retrieval_backend, '--resume'])
    CTX.save([str(P.labeled_pairs)])
else:
    print('No labeling export yet — see docs/vietnamese_labeling_guide.md. Skipping.')
